In [1]:
# 02_pricing_fairness_health.ipynb
# Insurance pricing / underwriting fairness (Medical Cost): proxy leakage under
# input restriction and effect-based (ex post) fairness testing (T-1, T-8).
# All code is contained in this single cell.

import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

def find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "results").is_dir() and (c / "notebooks").is_dir():
            return c
    return p.parent if p.name == "notebooks" else p
ROOT = find_root()
DATA, FIG, TAB = ROOT / "data", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85"})
G_DARK, G_LIGHT = "0.25", "0.65"
def savefig(fig, stem):
    for e in ("png", "pdf"):
        fig.savefig(FIG / (stem + "." + e), dpi=600, bbox_inches="tight")
    plt.close(fig)
def savetable(df, stem, index=False):
    df.to_csv(TAB / (stem + ".csv"), index=index)
    try:
        df.to_latex(TAB / (stem + ".tex"), index=index, float_format="%.3f")
    except Exception:
        pass

fp = DATA / "health_medcost.csv"
if not fp.exists():
    url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
    pd.read_csv(url).to_csv(fp, index=False)
df = pd.read_csv(fp)
y = df["charges"].values

# ---- (A) full pricing model vs sex-blind model: does the premium gap persist? ----
def price_cv(features, cat):
    pre = ColumnTransformer([
        ("num", StandardScaler(), [c for c in features if c not in cat]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    reg = Pipeline([("pre", pre), ("gb", GradientBoostingRegressor(random_state=SEED))])
    return cross_val_predict(reg, df[features], y, cv=KFold(5, shuffle=True, random_state=SEED))

feat_full = ["age", "sex", "bmi", "children", "smoker", "region"]
cat_full = ["sex", "smoker", "region"]
feat_blind = ["age", "bmi", "children", "smoker", "region"]
cat_blind = ["smoker", "region"]
pred_full = price_cv(feat_full, cat_full)
pred_blind = price_cv(feat_blind, cat_blind)

def group_gap(pred):
    m = pd.DataFrame({"sex": df["sex"], "pred": pred}).groupby("sex")["pred"].mean()
    return m, float(m.max() - m.min()), float(m.max() / m.min())
m_full, gap_full, ratio_full = group_gap(pred_full)
m_blind, gap_blind, ratio_blind = group_gap(pred_blind)
persist = pd.DataFrame({"model": ["with_sex", "sex_blind"],
    "mean_pred_female": [m_full.get("female"), m_blind.get("female")],
    "mean_pred_male": [m_full.get("male"), m_blind.get("male")],
    "abs_gap": [gap_full, gap_blind], "ratio": [ratio_full, ratio_blind]})
savetable(persist, "health_effect_persistence")

# ---- (B) effect-based test: disparate impact on "high premium" (top quartile) ----
thr = np.quantile(pred_blind, 0.75)
high = (pred_blind >= thr).astype(int)
di = pd.DataFrame({"sex": df["sex"], "high": high}).groupby("sex")["high"].mean()
di_ratio = float(di.min() / di.max())
reg_tab = pd.DataFrame({"group": di.index, "high_premium_rate": di.values})
reg_tab["disparate_impact_ratio"] = di_ratio
savetable(reg_tab, "health_effect_based_sex")
di_reg = pd.DataFrame({"region": df["region"], "high": high}).groupby(
    "region")["high"].mean().reset_index()
savetable(di_reg, "health_effect_based_region")

# ---- (C) proxy leakage: recover sex from sex-blind features ----
sx = (df["sex"] == "male").astype(int)
pre = ColumnTransformer([
    ("num", StandardScaler(), ["age", "bmi", "children"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["smoker", "region"])])
proxy = Pipeline([("pre", pre), ("gb", GradientBoostingClassifier(random_state=SEED))])
proxy_p = cross_val_predict(proxy, df[feat_blind], sx,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
proxy_auc = roc_auc_score(sx, proxy_p)
savetable(pd.DataFrame({"quantity": ["proxy_recovery_auc_from_sexblind_features", "chance"],
                        "value": [proxy_auc, 0.5]}), "health_proxy_auc")

# ---- figures ----
plot_df = pd.DataFrame({"sex": df["sex"], "Predicted charges": pred_blind})
fig, ax = plt.subplots(figsize=(4.2, 3.4))
sns.boxplot(data=plot_df, x="sex", y="Predicted charges", ax=ax,
            palette=[G_DARK, G_LIGHT], fliersize=2)
ax.set_xlabel("")
savefig(fig, "fig_health_charges_by_sex")

order = sorted(df["region"].unique())
plot_r = pd.DataFrame({"region": df["region"], "Predicted charges": pred_blind})
fig, ax = plt.subplots(figsize=(5.0, 3.4))
sns.boxplot(data=plot_r, x="region", y="Predicted charges", order=order, ax=ax,
            palette=sns.color_palette("Greys", len(order)), fliersize=2)
ax.set_xlabel("")
savefig(fig, "fig_health_charges_by_region")

fig, ax = plt.subplots(figsize=(4.0, 3.2))
sns.barplot(x=["sex-blind model"], y=[proxy_auc], ax=ax, color=G_DARK, edgecolor="black")
ax.axhline(0.5, color=G_LIGHT, ls="--", lw=1.2); ax.set_ylim(0, 1)
ax.set_ylabel("Proxy-recovery AUC (sex)"); ax.set_xlabel("")
savefig(fig, "fig_health_proxy_auc")

print("health leg done | sex-blind gap persists: abs=%.0f ratio=%.3f" % (gap_blind, ratio_blind),
      "| DI(high-premium) ratio: %.3f" % di_ratio, "| proxy AUC: %.3f" % proxy_auc)


health leg done | sex-blind gap persists: abs=1914 ratio=1.155 | DI(high-premium) ratio: 0.751 | proxy AUC: 0.445
